# LabOne Q — Lecture 2: Quantum Operations, Qubits, QPU and Topology

This notebook continues from Lecture 1, following the structure of the
[LabOne Q User Manual — Quantum Processing Unit](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/04_quantum_processing_unit/concepts/index.html):

1. **Quantum Operations** — reusable, named building blocks (`x180`, `measure`, ...) that build `Section`s
2. **Qubits** — the `QuantumElement`s that operations act on
3. **QPU** — the collection of qubits, operations and topology that describes the device physics
4. **Topology** — how qubits are connected, and how a two-qubit gate is defined on top of it

Everything below runs in **emulation mode** against a simulated demo device, so it can be executed without
being connected to real hardware.

## 1. Quantum Operations

A set of quantum operations is a class inheriting from `dsl.QuantumOperations` (see the
[Quantum Operations tutorial](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/04_quantum_processing_unit/tutorials/00_quantum_operations.html)).
Each method decorated with `@dsl.quantum_operation` takes the qubit(s) to act on as its first argument(s), and
plays pulses on their signal lines. `laboneq_applications` ships a ready-made set for tunable transmons:
`TunableTransmonOperations`.

We use `demo_platform` from `laboneq_applications` to get a `QuantumPlatform` — a `DeviceSetup`
plus a `QPU` — for two simulated transmons, with no hardware required.

In [ ]:
from laboneq.simple import *
from laboneq.contrib.example_helpers.plotting.plot_helpers import plot_simulation

from laboneq_applications.qpu_types.tunable_transmon.demo_qpus import demo_platform

platform = demo_platform(n_qubits=2)
qpu = platform.qpu
qop = qpu.quantum_operations
q0, q1 = qpu["q0"], qpu["q1"]

session = platform.session(do_emulation=True)
print("Connected:", session.connection_state.connected)

In [ ]:
print(qpu)

### Examining a set of operations

`qop.keys()` lists every registered operation. For any one of them, `.src` prints its underlying source —
the quickest way to see exactly what an operation will do without leaving the notebook.

In [ ]:
print(qop.keys())

In [ ]:
qop.rx?

### Calling an operation builds a Section

Calling an operation does not just describe a pulse — it **builds and returns a real `Section`**. The
section is named after the operation and the qubit(s) it acts on, and starts by reserving every signal
line of that qubit, so operations on the same qubit never overlap in time.

In [ ]:
section = qop.x180(q0)
section

## 2. Qubits

Operations act on **quantum elements** — for `TunableTransmonOperations`, these are `TunableTransmonQubit`
instances (see [Quantum Elements](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/04_quantum_processing_unit/tutorials/01_quantum_elements.html)).
A qubit bundles two things:

- `signals`: a dict mapping abstract line names (`drive`, `drive_ef`, `measure`, `acquire`, `flux`, ...) to
  logical signal paths on the `DeviceSetup`
- `parameters`: the physics/calibration numbers an operation reads (pulse amplitudes, lengths,
  frequencies, ...)

This is exactly what `qop.x180` used above: it read `q0.signals["drive"]` and
`q0.parameters.ge_drive_amplitude_pi` to build its pulse.

In [ ]:
print(q0.uid)
print(q0.signals)
print("pi-pulse amplitude:", q0.parameters.ge_drive_amplitude_pi)
print("readout frequency:", q0.parameters.readout_resonator_frequency)

In [ ]:
qpu["q0"].parameters

A qubit can also generate its own baseline `Calibration` from its parameters — this is what gets applied
to the `DeviceSetup` before compiling (`demo_platform` already did this for us).

In [ ]:
q0.calibration()

## 3. QPU

The [QPU](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/04_quantum_processing_unit/tutorials/02_qpu_topology.html)
ties everything together: the quantum elements, the operations that act on them, and the topology
connecting them. A `QPU` provides everything needed to *build* an experiment; the `DeviceSetup` (inside
`platform.setup`) is what is additionally needed to *compile* it for real hardware.

In [ ]:
print(qpu)

In [ ]:
print([q.uid for q in qpu.quantum_elements])
print(qpu["q0"] is q0)
print(qpu[:1])

## 4. Topology

The QPU topology is a directed graph: qubits (and other quantum elements, like couplers) sit at the
**nodes**, and **edges** describe how they are connected. Every edge has a user-defined `tag`, and may
optionally carry its own `QuantumParameters` and its own quantum element (e.g. the physical coupler).

`demo_platform(n_qubits=2)` gives us two independent qubits with no edges between them yet:

In [ ]:
qpu.topology.plot(disconnected=True, figsize=(3, 3));

### Building edges

Let's add a tunable coupler between `q0` and `q1`, and wire it into the topology as a `"cz"` edge. The edge
carries `CzParameters` — the coupler pulse and the single-qubit phase corrections a CZ gate needs — and
points at the coupler quantum element itself.

In [ ]:
from laboneq.dsl.quantum import QPU
from laboneq_applications.qpu_types.tunable_coupler import (
    TunableCoupler,
    TunableCouplerOperations,
    CzParameters,
)

coupler = TunableCoupler(uid="q0_q1_coupler", signals={"flux": "q0_q1_coupler/flux"})

# a QPU containing both qubits and the coupler, using the extended operation set that knows about "cz"
qpu_2q = QPU(
    quantum_elements=[q0, q1, coupler],
    quantum_operations=TunableCouplerOperations,
)

qpu_2q.topology.add_edge(
    "cz",
    "q0",
    "q1",
    parameters=CzParameters(
        coupler_pulse={"function": "const", "amplitude": 0.6, "length": 80e-9, "frequency": 0.0},
        control_angle=0.3,
        target_angle=-0.3,
    ),
    quantum_element=coupler,
)

qpu_2q.topology.add_edge(
    "cz",
    "q1",
    "q0",
    parameters=CzParameters(
        coupler_pulse={"function": "const", "amplitude": 0.6, "length": 80e-9, "frequency": 0.0},
        control_angle=0.3,
        target_angle=-0.3,
    ),
    quantum_element=coupler,
)

qpu_2q.topology.plot(figsize=(3, 3));

### A two-qubit gate built on the topology

`TunableCouplerOperations.cz` extends `TunableTransmonOperations` with a `cz` operation. Its source shows
it doing exactly what we just set up by hand: it looks up the `("cz", q0, q1)` edge, reads the coupler and
its `CzParameters` off it, plays the coupler pulse, then applies the Rz phase corrections.

In [ ]:
qop_2q = qpu_2q.quantum_operations
print(qop_2q.cz.src)

In [ ]:
cz_section = qop_2q.cz(qpu_2q["q0"], qpu_2q["q1"])
cz_section

In [ ]:
qpu_2q.topology["cz", :, :]

## 5. Operations in a real experiment

Finally, a couple of operations strung together inside an `acquire_loop_rt` are already a (very short)
real experiment: compile it against the platform's session and look at the simulated output.

In [ ]:
exp = Experiment(
    uid="ops_demo",
    signals=[ExperimentSignal(name, map_to=path) for name, path in q0.signals.items()],
)
with exp.acquire_loop_rt(count=1, averaging_mode=AveragingMode.SEQUENTIAL):
    qop.x180(q0)
    qop.measure(q0, handle="result")
    qop.passive_reset(q0)

compiled_exp = session.compile(exp)
plot_simulation(compiled_exp, length=20e-9)

In [ ]:
qop.x180(qpu["q0"])